# Stage 7E: spatially robust public blend gate

This confirmation run fixes the public package branch to its postprocessed OOF. For each outer fold/block, a 20/30/40% weight is eligible only when it does not worsen any inner fold/block. It does not submit.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess

REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir = drive_root / 'artifacts'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
python = repo_dir / '.venv/bin/python'
subprocess.run(['uv', 'pip', 'install', '--python', str(python), 'kagglehub'], check=True)
artifact_dir.mkdir(parents=True, exist_ok=True)
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)

In [ ]:
download = subprocess.run([
    str(python), '-c',
    "import kagglehub; print('DATASET_PATH=' + kagglehub.dataset_download('pilkwang/rogii-model-package'))"
], check=True, capture_output=True, text=True)
line = [value for value in download.stdout.splitlines() if value.startswith('DATASET_PATH=')][-1]
package_dir = Path(line.split('=', 1)[1])
assert (package_dir / 'oof/blend_oof_postprocessed.npy').is_file(), package_dir
print('Package:', package_dir)

In [ ]:
BASE_RUN_ID = 'stage7_public_residual_gate_full_v001'
RUN_ID = 'stage7e_public_robust_blend_full_v001'
base_run = artifact_dir / BASE_RUN_ID
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'gate_summary.json').is_file():
    subprocess.run([
        'uv', 'run', 'rogii-public-blend',
        '--config', 'configs/experiment/public_robust_blend_gate.yaml',
        '--base-run', str(base_run),
        '--package-dir', str(package_dir),
        '--artifact-dir', str(artifact_dir),
        '--run-id', RUN_ID,
    ], cwd=repo_dir, check=True)
else:
    print('Reusing completed run:', run_dir)

In [ ]:
gate = json.loads((run_dir / 'gate_summary.json').read_text())
standard = json.loads((run_dir / 'standard_selections.json').read_text())
spatial = json.loads((run_dir / 'spatial_selections.json').read_text())
manifest = json.loads((run_dir / 'blend_manifest.json').read_text())
{
    'promoted': gate['promoted'],
    'base_rmse': gate['base_metrics']['pooled_rmse'],
    'robust_nested_rmse': gate['candidate_metrics']['pooled_rmse'],
    'rmse_delta': gate['pooled_rmse_delta'],
    'bootstrap_95pct': [gate['bootstrap']['ci_2_5'], gate['bootstrap']['ci_97_5']],
    'improved_folds': f"{gate['improved_folds']}/{len(gate['fold_deltas'])}",
    'standard_fold_deltas': gate['fold_deltas'],
    'gates': gate['gates'],
    'spatial_delta': gate['spatial']['pooled_rmse_delta'],
    'spatial_improved_folds': gate['spatial']['improved_folds'],
    'spatial_required_folds': gate['spatial']['required_improved_folds'],
    'spatial_fold_deltas': gate['spatial']['fold_deltas'],
    'standard_selections': standard,
    'spatial_selections': spatial,
    'inference_spec': manifest['inference_spec'],
}

Only `promoted: True` authorizes Kaggle integration. This is a robustness confirmation on the same OOF population, not a new independent dataset, so the eventual Kaggle change remains conservative and branch-local.